In [57]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import seaborn as sns
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    mutual_info_score
)
from sklearn.model_selection import KFold





In [58]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("hopesb/student-depression-dataset",
    output_dir="./student_depression_dataset")

print("Path to dataset files:", path)




Path to dataset files: ./student_depression_dataset


In [59]:
df= pd.read_csv(path+"/Student Depression Dataset.csv")
df.head()

,id,Gender,Age,City,Profession,Academic Pressure,Work Pressure,CGPA,Study Satisfaction,Job Satisfaction,Sleep Duration,Dietary Habits,Degree,Have you ever had suicidal thoughts ?,Work/Study Hours,Financial Stress,Family History of Mental Illness,Depression
0,2,Male,33.0,Visakhapatnam,Student,5.0,0.0,8.97,2.0,0.0,5-6 hours,Healthy,B.Pharm,Yes,3.0,1.0,No,1
1,8,Female,24.0,Bangalore,Student,2.0,0.0,5.90,5.0,0.0,5-6 hours,Moderate,BSc,No,3.0,2.0,Yes,0
2,26,Male,31.0,Srinagar,Student,3.0,0.0,7.03,5.0,0.0,Less than 5 hours,Healthy,BA,No,9.0,1.0,Yes,0
3,30,Female,28.0,Varanasi,Student,3.0,0.0,5.59,2.0,0.0,7-8 hours,Moderate,BCA,Yes,4.0,5.0,Yes,1
4,32,Female,25.0,Jaipur,Student,4.0,0.0,8.13,3.0,0.0,5-6 hours,Moderate,M.Tech,Yes,1.0,1.0,No,0


In [60]:
df.columns=df.columns.str.lower().str.replace(" ", "_")
for col in df.columns:
    if (df[col].dtype=="str"):
        df[col]= df[col].str.lower().str.replace(" ","_")



In [61]:
del df["id"]
del df["profession"]
del df["work_pressure"]
del df["job_satisfaction"]
del df["city"]

In [62]:
df=df.dropna().reset_index(drop=True)
categorical_columns=["gender","sleep_duration","dietary_habits","degree","have_you_ever_had_suicidal_thoughts_?","family_history_of_mental_illness"]
numerical_columns=["age","academic_pressure","cgpa","study_satisfaction","work/study_hours","financial_stress"]
df.head()

,gender,age,academic_pressure,cgpa,study_satisfaction,sleep_duration,dietary_habits,degree,have_you_ever_had_suicidal_thoughts_?,work/study_hours,financial_stress,family_history_of_mental_illness,depression
0,male,33.0,5.0,8.97,2.0,5-6_hours,healthy,b.pharm,yes,3.0,1.0,no,1
1,female,24.0,2.0,5.90,5.0,5-6_hours,moderate,bsc,no,3.0,2.0,yes,0
2,male,31.0,3.0,7.03,5.0,less_than_5_hours,healthy,ba,no,9.0,1.0,yes,0
3,female,28.0,3.0,5.59,2.0,7-8_hours,moderate,bca,yes,4.0,5.0,yes,1
4,female,25.0,4.0,8.13,3.0,5-6_hours,moderate,m.tech,yes,1.0,1.0,no,0


In [63]:
from sklearn.model_selection import train_test_split

df_full_train, df_test=train_test_split(df, test_size=0.2, random_state=5234)
df_train, df_val=train_test_split(df_full_train, test_size=0.25, random_state=5234)
df_train = df_train.reset_index(drop=True)
df_val = df_val.reset_index(drop=True)
df_test = df_test.reset_index(drop=True)

In [64]:
y_train=df_train["depression"]
y_val=df_val["depression"]
y_test=df_test["depression"]

In [65]:
del df_train["depression"]
del df_val["depression"]
del df_test["depression"]

In [66]:
def train_logistic_regression(df_train:pd.DataFrame, y_train, c):
    dicts=df_train.to_dict(orient="records")
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(dicts)
    model = LogisticRegression(max_iter=1000, C=c)
    model.fit(X_train, y_train)
    return dv, model

def predict_logistic_regression(df, dv, model):
    dicts = df.to_dict(orient='records')
    X = dv.transform(dicts)
    y_pred = model.predict_proba(X)[:, 1]
    return y_pred




In [67]:
for c in [0.1,0.5,1,2,3,4]:
    dv,model=train_logistic_regression(df_train, y_train, c)
    y_predict=predict_logistic_regression(df_val, dv, model)
    print(roc_auc_score(y_val, y_predict))

0.9225917989805287
0.9223875054899717
0.9223666103251396
0.9222991335826557
0.9222600050574286
0.922269986824068


In [68]:
kfold= KFold(n_splits=5,shuffle=True, random_state=2344)

scores=[]

for train_idx, val_idx in kfold.split(df_full_train):
    df_train=df_full_train.iloc[train_idx]
    df_val=df_full_train.iloc[val_idx]
    y_train = df_train.depression.values
    y_val = df_val.depression.values
    df_train=df_train.drop(columns=["depression"])
    df_val=df_val.drop(columns=["depression"])
    dv, model = train_logistic_regression(df_train, y_train,1)
    y_pred = predict_logistic_regression(df_val, dv, model)
    auc = roc_auc_score(y_val, y_pred)
    scores.append(auc)



In [69]:
scores=np.array(scores)
print(scores.mean())

0.920038427291707


In [70]:
dv, final_model=train_logistic_regression(df_full_train.drop(columns=["depression"]), df_full_train["depression"], 1.0)
y_pred=predict_logistic_regression(df_test, dv, final_model)


In [71]:
df_full_train

,gender,age,academic_pressure,cgpa,study_satisfaction,sleep_duration,dietary_habits,degree,have_you_ever_had_suicidal_thoughts_?,work/study_hours,financial_stress,family_history_of_mental_illness,depression
17270,female,29.0,5.0,5.79,4.0,7-8_hours,moderate,m.ed,yes,5.0,5.0,no,1
18778,female,31.0,4.0,5.79,4.0,more_than_8_hours,moderate,m.pharm,yes,6.0,5.0,no,1
21512,female,28.0,3.0,8.54,2.0,5-6_hours,healthy,bba,yes,10.0,3.0,yes,1
24844,female,26.0,3.0,9.01,2.0,5-6_hours,moderate,llb,yes,10.0,3.0,no,1
15452,male,19.0,2.0,7.68,2.0,less_than_5_hours,moderate,class_12,yes,5.0,2.0,no,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
26614,male,34.0,2.0,8.70,3.0,more_than_8_hours,unhealthy,md,yes,7.0,5.0,yes,1
16425,female,31.0,4.0,9.92,3.0,5-6_hours,moderate,llb,yes,11.0,5.0,yes,1
4262,male,25.0,5.0,9.94,3.0,5-6_hours,unhealthy,bca,no,10.0,2.0,no,0
18256,female,27.0,2.0,5.12,4.0,less_than_5_hours,unhealthy,bhm,yes,9.0,3.0,yes,0


In [72]:
y_pred_class = (y_pred >= 0.5).astype(int)


accuracy = accuracy_score(y_test, y_pred_class)
recall = recall_score(y_test, y_pred_class)
f1 = f1_score(y_test, y_pred_class)
auc = roc_auc_score(y_test, y_pred)
precision = precision_score(y_test, y_pred_class)


print("Accuracy:  ", round(accuracy, 4))
print("Recall:    ", round(recall, 4))
print("F1 Score:  ", round(f1, 4))
print("AUC:       ", round(auc, 4))
print("Precision: ", round(precision, 4))

Accuracy:   0.8525
Recall:     0.8912
F1 Score:   0.8757
AUC:        0.9238
Precision:  0.8607
